In [8]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

data_dir = root / "data"
fig_dir = root / "figures"
data_dir.mkdir(exist_ok=True)
fig_dir.mkdir(exist_ok=True)

data_dir = Path("..") / "data"
train = pd.read_parquet(data_dir / "train.parquet")
test = pd.read_parquet(data_dir / "test.parquet")  # transferring the test and train data to the notebook for analysis

In [9]:
train["drivage_bands"] = pd.cut(train["DrivAge"], bins=[18, 25, 35, 45, 55, 65, 75, 85, 100], labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-85", "86-100"], include_lowest=True)
test["drivage_bands"] = pd.cut(test["DrivAge"], bins=[18, 25, 35, 45, 55, 65, 75, 85, 100], labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-85", "86-100"], include_lowest=True)
train["vehage_bands"] = pd.cut(train["VehAge"], bins=[0, 1, 3, 5, 7 ,10, 15, 20], labels=["0-1","2-3", "4-5", "6-7", "8-10", "11-15", "16-20"], include_lowest=True)
test["vehage_bands"] = pd.cut(test["VehAge"], bins=[0, 1, 3, 5, 7 ,10, 15, 20], labels=["0-1","2-3", "4-5", "6-7", "8-10", "11-15", "16-20"], include_lowest=True)

In [31]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.glm(
    "ClaimNb ~ C(drivage_bands) + C(vehage_bands) + np.log(Density) + BonusMalus + C(Region) + C(VehBrand) + C(VehGas)",
    data=train,
    family=sm.families.Poisson(),
    offset=np.log(train["Exposure"]),
).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               534794
Model:                            GLM   Df Residuals:                   534746
Model Family:                 Poisson   Df Model:                           47
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -84892.
Date:                Mon, 09 Nov 2026   Deviance:                   1.2953e+05
Time:                        18:51:26   Pearson chi2:                 8.77e+05
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01064
Covariance Type:            nonrobust                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

In [15]:
test["PredCount"] = model.predict(test, offset=np.log(test["Exposure"]))
test["PredFreq"] = test["PredCount"] / test["Exposure"]
test.sort_values("PredCount")

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,TotalClaimAmount,drivage_bands,vehage_bands,PredCount,PredFreq
371102,3030159.0,0,0.002740,C,11,1,33,50,B12,'Regular',162,R73,0.0,26-35,0-1,0.000064,0.023184
579958,4171136.0,0,0.002732,A,9,17,30,57,B1,'Regular',30,R24,0.0,26-35,16-20,0.000082,0.029981
541703,4132881.0,0,0.002732,C,4,19,30,50,B2,'Regular',282,R72,0.0,26-35,16-20,0.000089,0.032565
515202,4066523.0,0,0.002732,C,8,0,67,50,B12,'Regular',116,R41,0.0,66-75,0-1,0.000091,0.033317
920,1921.0,0,0.002732,A,4,0,63,50,B12,'Regular',41,R91,0.0,56-65,0-1,0.000094,0.034271
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284618,2182249.0,1,1.000000,D,4,3,79,158,B1,'Regular',1182,R82,1172.0,76-85,2-3,1.302316,1.302316
563447,4154625.0,2,1.000000,E,6,5,20,156,B6,'Diesel',4762,R93,2408.0,18-25,4-5,1.318952,1.318952
178152,1177433.0,0,0.910000,C,5,2,31,185,B3,'Diesel',473,R53,0.0,26-35,2-3,1.609909,1.769131
396261,3075586.0,0,1.000000,E,7,13,83,173,B2,'Regular',3688,R82,0.0,76-85,11-15,1.853167,1.853167


In [28]:
from sklearn.metrics import mean_poisson_deviance

base_frequency = train["ClaimNb"].sum() / train["Exposure"].sum()
train_deviance = mean_poisson_deviance((train["ClaimNb"] / train["Exposure"]), [base_frequency] * len(train), sample_weight = train["Exposure"])
test_deviance = mean_poisson_deviance((test["ClaimNb"] / test["Exposure"]), [base_frequency] * len(test), sample_weight = test["Exposure"])

glm_deviance = mean_poisson_deviance((test["ClaimNb"] / test["Exposure"]), test["PredFreq"], sample_weight = test["Exposure"])
glm_deviance_ratio = glm_deviance / test_deviance
print(test_deviance, glm_deviance, f"{1-glm_deviance_ratio:.2%}")

split_deviance_ratio = train_deviance / test_deviance
print(test_deviance, train_deviance, f"{1-split_deviance_ratio:.2%}")



0.47958347067937185 0.45898578471860424 4.29%
0.47958347067937185 0.4802493321652381 -0.14%


In [29]:
print(np.exp(model.params))

Intercept                     0.009661
C(drivage_bands)[T.26-35]     0.857431
C(drivage_bands)[T.36-45]     1.117924
C(drivage_bands)[T.46-55]     1.267805
C(drivage_bands)[T.56-65]     1.131020
C(drivage_bands)[T.66-75]     1.100617
C(drivage_bands)[T.76-85]     1.205844
C(drivage_bands)[T.86-100]    1.407065
C(vehage_bands)[T.2-3]        1.035055
C(vehage_bands)[T.4-5]        1.015777
C(vehage_bands)[T.6-7]        1.078494
C(vehage_bands)[T.8-10]       1.000728
C(vehage_bands)[T.11-15]      0.917235
C(vehage_bands)[T.16-20]      0.765132
C(Region)[T.R21]              1.012527
C(Region)[T.R22]              1.152980
C(Region)[T.R23]              0.937232
C(Region)[T.R24]              0.982527
C(Region)[T.R25]              0.961599
C(Region)[T.R26]              1.021775
C(Region)[T.R31]              1.060161
C(Region)[T.R41]              0.903960
C(Region)[T.R42]              0.913356
C(Region)[T.R43]              0.990668
C(Region)[T.R52]              1.003279
C(Region)[T.R53]         